## import important libs

In [42]:
import $ivy.`org.apache.spark:spark-core_2.12:3.5.3`
import $ivy.`org.apache.spark:spark-sql_2.12:3.5.3`
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.types._
import org.apache.spark.sql.expressions.Window

import $ivy.$                                       

import $ivy.$                                      

import org.apache.spark.sql.SparkSession

import org.apache.spark.sql.functions._

import org.apache.spark.sql.DataFrame

import org.apache.spark.sql.types._

import org.apache.spark.sql.expressions.Window

## creating spark application

In [43]:
val spark = SparkSession.builder().master("local[*]").appName("readDF").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

spark: SparkSession = org.apache.spark.sql.SparkSession@37f8ca76
import spark.implicits._

## Reading Spark DF

In [44]:
  val studentSchema: StructType = StructType(Array(
    StructField("student_id", StringType, nullable = false),
    StructField("name", StringType, nullable = false),
    StructField("age", IntegerType, nullable = false),
    StructField("gender", StringType, nullable = false),
    StructField("city", StringType, nullable = false),
    StructField("enrollment_year", IntegerType, nullable = true),
    StructField("university_id", StringType, nullable = true)
  ))

  println("Reading the student csv file...")
  val studentDf: DataFrame = spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .schema(studentSchema)
    .load("resource/data/student.csv")
  studentDf.show()

Reading the student csv file...
+----------+-----------+---+------+---------+---------------+-------------+
|student_id|       name|age|gender|     city|enrollment_year|university_id|
+----------+-----------+---+------+---------+---------------+-------------+
|      S001|Amit Sharma| 20|     M|    Delhi|           2022|          U01|
|      S002| Neha Verma| 21|     F|   Mumbai|           2021|          U02|
|      S003|Rahul Singh| 19|     M|  Lucknow|           2023|          U01|
|      S004|Priya Mehta| 22|     F|    Delhi|           2020|          U03|
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|
|      S007|Vikas Gupta| 23|     M|   Jaipur|           2019|          U01|
|      S008| Kavya Nair| 20|     F|    Kochi|           2022|          U02|
|      S009|  Rohan Das| 22|     M|  Kolkata|           2020|          U03|
|      S010|Ananya Iyer| 21|     F|  Chennai|           

studentSchema: StructType = StructType(
  StructField("student_id", StringType, false, {}),
  StructField("name", StringType, false, {}),
  StructField("age", IntegerType, false, {}),
  StructField("gender", StringType, false, {}),
  StructField("city", StringType, false, {}),
  StructField("enrollment_year", IntegerType, true, {}),
  StructField("university_id", StringType, true, {})
)
studentDf: DataFrame = [student_id: string, name: string ... 5 more fields]

## Basic Operation
### 1. Select
The `select` method allows you to transform, select, and manipulate specific columns from a DataFrame. It is equivalent to a SQL `SELECT` statement. You can pass string column names or Column objects.  
**Select a single column:**

In [45]:
studentDf.select(col("student_id")).limit(2).show()

+----------+
|student_id|
+----------+
|      S001|
|      S002|
+----------+



**select multiple column:**

In [46]:
studentDf.select(col("student_id"), col("name")).limit(2).show()

+----------+-----------+
|student_id|       name|
+----------+-----------+
|      S001|Amit Sharma|
|      S002| Neha Verma|
+----------+-----------+



**select Using expressions (selectExpr):**
* For complex expressions (like renaming or SQL-like logic), `selectExpr` is often more convenient.

In [47]:
studentDf.selectExpr("*", "age<20 as below_20").show()

+----------+-----------+---+------+---------+---------------+-------------+--------+
|student_id|       name|age|gender|     city|enrollment_year|university_id|below_20|
+----------+-----------+---+------+---------+---------------+-------------+--------+
|      S001|Amit Sharma| 20|     M|    Delhi|           2022|          U01|   false|
|      S002| Neha Verma| 21|     F|   Mumbai|           2021|          U02|   false|
|      S003|Rahul Singh| 19|     M|  Lucknow|           2023|          U01|    true|
|      S004|Priya Mehta| 22|     F|    Delhi|           2020|          U03|   false|
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|   false|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|   false|
|      S007|Vikas Gupta| 23|     M|   Jaipur|           2019|          U01|   false|
|      S008| Kavya Nair| 20|     F|    Kochi|           2022|          U02|   false|
|      S009|  Rohan Das| 22|     M|  Kolkata|           2020|    

### Where (and Filter)
The `where` method filters rows based on a condition that evaluates to true or false. In Spark, `where` and `filter` are synonymous and can be used interchangeably.

In [48]:
studentDf.where(col("age")<20).show()

+----------+-----------+---+------+-------+---------------+-------------+
|student_id|       name|age|gender|   city|enrollment_year|university_id|
+----------+-----------+---+------+-------+---------------+-------------+
|      S003|Rahul Singh| 19|     M|Lucknow|           2023|          U01|
+----------+-----------+---+------+-------+---------------+-------------+



In [49]:
studentDf.filter(col("age")<20).show()

+----------+-----------+---+------+-------+---------------+-------------+
|student_id|       name|age|gender|   city|enrollment_year|university_id|
+----------+-----------+---+------+-------+---------------+-------------+
|      S003|Rahul Singh| 19|     M|Lucknow|           2023|          U01|
+----------+-----------+---+------+-------+---------------+-------------+



### Distinct
The `distinct` method deduplicates rows in a DataFrame, returning a new DataFrame with only unique rows.

In [50]:
studentDf.selectExpr("student_id", "name").distinct().show(5)

+----------+-----------+
|student_id|       name|
+----------+-----------+
|      S008| Kavya Nair|
|      S002| Neha Verma|
|      S003|Rahul Singh|
|      S005|Arjun Patel|
|      S010|Ananya Iyer|
+----------+-----------+
only showing top 5 rows



### OrderBy (and Sort)
The `orderBy` and `sort` methods are used to sort the rows in a DataFrame. By default, they sort in ascending order. To specify direction, you can use the `asc` and `desc` functions.

In [51]:
// using sort
studentDf.sort("student_id").show()

+----------+-----------+---+------+---------+---------------+-------------+
|student_id|       name|age|gender|     city|enrollment_year|university_id|
+----------+-----------+---+------+---------+---------------+-------------+
|      S001|Amit Sharma| 20|     M|    Delhi|           2022|          U01|
|      S002| Neha Verma| 21|     F|   Mumbai|           2021|          U02|
|      S003|Rahul Singh| 19|     M|  Lucknow|           2023|          U01|
|      S004|Priya Mehta| 22|     F|    Delhi|           2020|          U03|
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|
|      S007|Vikas Gupta| 23|     M|   Jaipur|           2019|          U01|
|      S007|Vikas Gupta| 23|     M|   Jaipur|           2019|          U01|
|      S008|

In [52]:
// using orderBy()
studentDf.orderBy(desc("Student_id")).show()

+----------+-----------+---+------+---------+---------------+-------------+
|student_id|       name|age|gender|     city|enrollment_year|university_id|
+----------+-----------+---+------+---------+---------------+-------------+
|      S010|Ananya Iyer| 21|     F|  Chennai|           2021|          U01|
|      S010|Ananya Iyer| 21|     F|  Chennai|           2021|          U01|
|      S009|  Rohan Das| 22|     M|  Kolkata|           2020|          U03|
|      S009|  Rohan Das| 22|     M|  Kolkata|           2020|          U03|
|      S008| Kavya Nair| 20|     F|    Kochi|           2022|          U02|
|      S008| Kavya Nair| 20|     F|    Kochi|           2022|          U02|
|      S007|Vikas Gupta| 23|     M|   Jaipur|           2019|          U01|
|      S007|Vikas Gupta| 23|     M|   Jaipur|           2019|          U01|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|
|      S005|

### Limit
The `limit` method restricts the number of rows extracted from the DataFrame. It is functionally equivalent to the SQL `LIMIT` clause.

In [53]:
studentDf.select("*").limit(5).show()

+----------+-----------+---+------+---------+---------------+-------------+
|student_id|       name|age|gender|     city|enrollment_year|university_id|
+----------+-----------+---+------+---------+---------------+-------------+
|      S001|Amit Sharma| 20|     M|    Delhi|           2022|          U01|
|      S002| Neha Verma| 21|     F|   Mumbai|           2021|          U02|
|      S003|Rahul Singh| 19|     M|  Lucknow|           2023|          U01|
|      S004|Priya Mehta| 22|     F|    Delhi|           2020|          U03|
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|
+----------+-----------+---+------+---------+---------------+-------------+



### Adding default value as new column in select
* We need to use lit()
* lit() = convert a constant value → Spark Column

In [54]:
studentDf.select(col("*"), lit("DefailtValue") as "default").show(2)

// we can create alias like below also
studentDf.select(col("*"), lit("DefailtValue").alias("value")).show(2)

+----------+-----------+---+------+------+---------------+-------------+------------+
|student_id|       name|age|gender|  city|enrollment_year|university_id|     default|
+----------+-----------+---+------+------+---------------+-------------+------------+
|      S001|Amit Sharma| 20|     M| Delhi|           2022|          U01|DefailtValue|
|      S002| Neha Verma| 21|     F|Mumbai|           2021|          U02|DefailtValue|
+----------+-----------+---+------+------+---------------+-------------+------------+
only showing top 2 rows

+----------+-----------+---+------+------+---------------+-------------+------------+
|student_id|       name|age|gender|  city|enrollment_year|university_id|       value|
+----------+-----------+---+------+------+---------------+-------------+------------+
|      S001|Amit Sharma| 20|     M| Delhi|           2022|          U01|DefailtValue|
|      S002| Neha Verma| 21|     F|Mumbai|           2021|          U02|DefailtValue|
+----------+-----------+---+-

## Aggregation Function
### count
* The first function worth going over is count, except in this example it will perform as a transformation instead of an action. 
* In this case, we can do one of two things: specify a specific column to count, 
* or all the columns by using count(*) or count(1) to represent that we want to count every row as the literal one, as shown in this example

In [55]:
// count all column => count() => return long
studentDf.count()

// count of stundet_id 
studentDf.select(count("student_id")).show()

// count *
studentDf.select(count("*")).show()



+-----------------+
|count(student_id)|
+-----------------+
|               16|
+-----------------+

+--------+
|count(1)|
+--------+
|      16|
+--------+



res54_0: Long = 16L

There are a number of gotchas when it comes to null values and counting. For instance, when
performing a count(*), Spark will count null values (including rows containing all nulls). However,
when counting an individual column, Spark will not count the null values

### countDistinct()
* used to count the distinct value for the column

In [56]:
// count the unique value
studentDf.select(countDistinct("student_id")).show()

+--------------------------+
|count(DISTINCT student_id)|
+--------------------------+
|                        10|
+--------------------------+



### min and max

In [57]:
// minimun age
studentDf.select(min("age")).show()

// max age
studentDf.select(max("age")).show()

// min and max together
studentDf.select(min("age") as "min_age", max("age") as "max_age").show()

+--------+
|min(age)|
+--------+
|      19|
+--------+

+--------+
|max(age)|
+--------+
|      23|
+--------+

+-------+-------+
|min_age|max_age|
+-------+-------+
|     19|     23|
+-------+-------+



### sum and sumDistinct

In [58]:
// sum of age of students
studentDf.select(sum("age")).show()

// sumDistinct
studentDf.select(sumDistinct("age")).show()

+--------+
|sum(age)|
+--------+
|     336|
+--------+

+-----------------+
|sum(DISTINCT age)|
+-----------------+
|              105|
+-----------------+



### groupBy
* Grouping is used to perform calculations on subsets of data rather than the entire DataFrame.
* Typically applied on categorical columns (e.g., class, category, invoice number).
* After grouping, aggregation functions (count, sum, avg, etc.) are applied to each group.
* This is one of the most common operations in data analytics and ETL pipelines

#### Two Phases of Grouping
Grouping in Spark happens in two logical steps:
* **1️⃣ Specify grouping columns**
    * Using groupBy()
    * Returns a RelationalGroupedDataset
* **2️⃣ Apply aggregation**
    * Using functions like count, sum, avg, or agg()
    * Returns a DataFrame

In [59]:
studentDf.groupBy("student_id") // 👉 It returns a RelationalGroupedDataset (a grouped object), which is an intermediate representation used to define aggregations.

res58: org.apache.spark.sql.RelationalGroupedDataset = RelationalGroupedDataset: [grouping expressions: [student_id: string], value: [student_id: string, name: string ... 5 more fields], type: GroupBy]

In [60]:
// group by on column with aggregate function
studentDf.groupBy("student_id").count().show()

+----------+-----+
|student_id|count|
+----------+-----+
|      S004|    1|
|      S001|    1|
|      S008|    2|
|      S009|    2|
|      S002|    1|
|      S005|    2|
|      S006|    2|
|      S003|    1|
|      S007|    2|
|      S010|    2|
+----------+-----+



In [61]:
// retrun the diplicate students using student_id
studentDf.groupBy("student_id").count().filter(col("count")>1).show()

+----------+-----+
|student_id|count|
+----------+-----+
|      S008|    2|
|      S009|    2|
|      S005|    2|
|      S006|    2|
|      S007|    2|
|      S010|    2|
+----------+-----+



### Grouping with Expressions
* count() method is a shortcut, but using aggregation expressions is more flexible.
* The agg() function allows:
    * Multiple aggregations
    * Custom expressions
    * Aliasing columns

In [62]:
studentDf.groupBy("student_id").agg(avg("age") as "avg_age", count("student_id") as "count").show()

+----------+-------+-----+
|student_id|avg_age|count|
+----------+-------+-----+
|      S004|   22.0|    1|
|      S001|   20.0|    1|
|      S008|   20.0|    2|
|      S009|   22.0|    2|
|      S002|   21.0|    1|
|      S005|   20.0|    2|
|      S006|   21.0|    2|
|      S003|   19.0|    1|
|      S007|   23.0|    2|
|      S010|   21.0|    2|
+----------+-------+-----+



### Grouping with Maps
Sometimes, it can be easier to specify your transformations as a series of Maps for which the key
is the column, and the value is the aggregation function (as a string) that you would like to
perform. You can reuse multiple column names if you specify them inline, as well:

In [63]:
studentDf.groupBy("student_id").agg("age" -> "avg", "student_id" -> "count").show()

+----------+--------+-----------------+
|student_id|avg(age)|count(student_id)|
+----------+--------+-----------------+
|      S004|    22.0|                1|
|      S001|    20.0|                1|
|      S008|    20.0|                2|
|      S009|    22.0|                2|
|      S002|    21.0|                1|
|      S005|    20.0|                2|
|      S006|    21.0|                2|
|      S003|    19.0|                1|
|      S007|    23.0|                2|
|      S010|    21.0|                2|
+----------+--------+-----------------+



## Window Function
* A window function calculates a return value for every input row of a DataFrame based on a specific group of rows, called a **frame**

It is distinct from a standard groupBy aggregation in the following way:
* **GroupBy:** Collapses all rows in a group into a single result row (e.g., calculating the total sales per country returns one row per country).
* **Window Function:** Maintains the original number of rows but appends a new column with a calculated value for each row based on its group (e.g., calculating the rank of a specific sale within a country, or a running total)

### Core Components of a Window Specification
1.  **Partitioning (`partitionBy`):** This controls how rows are grouped. It is conceptually similar to a `groupBy`, isolating calculations to specific groups (e.g., calculating ranks *per customer*).
2.  **Ordering (`orderBy`):** This determines the order of rows within a partition. Ordering is essential for functions like ranking or calculating cumulative totals.
3.  **Frame Specification (`rowsBetween` / `rangeBetween`):** This defines exactly which rows are included in the calculation for the current row (e.g., "all previous rows up to the current row" for a running total).

### Types of Window Functions
Spark supports three categories of window functions:
| Category | Functions | Description |
| :--- | :--- | :--- |
| **Ranking** | `rank`, `dense_rank`, `percent_rank`, `ntile`, `row_number` | Assigns a rank to rows within a partition based on the specified order. |
| **Analytic** | `cume_dist`, `first_value`, `last_value`, `lag`, `lead` | Accesses data from other rows in the window (e.g., `lag` gets the value from the previous row). |
| **Aggregate** | `sum`, `count`, `avg`, `min`, `max` | Performs standard aggregations over the defined window frame. |


In [64]:
// student df have duplicates so find duplicates row based on student_id

val windowFunc = Window
    .partitionBy("student_id")
    .orderBy(col("name").desc)

studentDf.withColumn("rn", row_number().over(windowFunc)).filter(col("rn")>1).show()

+----------+-----------+---+------+---------+---------------+-------------+---+
|student_id|       name|age|gender|     city|enrollment_year|university_id| rn|
+----------+-----------+---+------+---------+---------------+-------------+---+
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|  2|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|  2|
|      S007|Vikas Gupta| 23|     M|   Jaipur|           2019|          U01|  2|
|      S008| Kavya Nair| 20|     F|    Kochi|           2022|          U02|  2|
|      S009|  Rohan Das| 22|     M|  Kolkata|           2020|          U03|  2|
|      S010|Ananya Iyer| 21|     F|  Chennai|           2021|          U01|  2|
+----------+-----------+---+------+---------+---------------+-------------+---+



windowFunc: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@2e337032

In [65]:
// remove the duplicate based on student_id and keep onlu the unique records.
studentDf.withColumn("rn", row_number().over(Window.partitionBy("student_id").orderBy("name"))).filter(col("rn")===1).show()

+----------+-----------+---+------+---------+---------------+-------------+---+
|student_id|       name|age|gender|     city|enrollment_year|university_id| rn|
+----------+-----------+---+------+---------+---------------+-------------+---+
|      S001|Amit Sharma| 20|     M|    Delhi|           2022|          U01|  1|
|      S002| Neha Verma| 21|     F|   Mumbai|           2021|          U02|  1|
|      S003|Rahul Singh| 19|     M|  Lucknow|           2023|          U01|  1|
|      S004|Priya Mehta| 22|     F|    Delhi|           2020|          U03|  1|
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|  1|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|  1|
|      S007|Vikas Gupta| 23|     M|   Jaipur|           2019|          U01|  1|
|      S008| Kavya Nair| 20|     F|    Kochi|           2022|          U02|  1|
|      S009|  Rohan Das| 22|     M|  Kolkata|           2020|          U03|  1|
|      S010|Ananya Iyer| 21|     F|  Che

In [66]:
// calcualte the running sum of age of student based on city
studentDf.withColumn("running_sum", sum(col("age")).over(Window.partitionBy(col("city")).orderBy(desc(("name"))))).show(100)

+----------+-----------+---+------+---------+---------------+-------------+-----------+
|student_id|       name|age|gender|     city|enrollment_year|university_id|running_sum|
+----------+-----------+---+------+---------+---------------+-------------+-----------+
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|         40|
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|         40|
|      S010|Ananya Iyer| 21|     F|  Chennai|           2021|          U01|         42|
|      S010|Ananya Iyer| 21|     F|  Chennai|           2021|          U01|         42|
|      S004|Priya Mehta| 22|     F|    Delhi|           2020|          U03|         22|
|      S001|Amit Sharma| 20|     M|    Delhi|           2022|          U01|         42|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|         42|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|         42|
|      S007|Vikas Gupta| 23|    

> **💡-Note**
> * if you see in above output the `running_sum` is not working properly when the order by value is same(duplicate value).   
> * for all duplicate value we are getting the same `running_sum`

In [67]:
val windowSpecs = Window
    .partitionBy(col("city"))
    .orderBy(col("name").desc)
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

studentDf.withColumn("running_sum", sum(col("age")).over(windowSpecs)).show(100)

+----------+-----------+---+------+---------+---------------+-------------+-----------+
|student_id|       name|age|gender|     city|enrollment_year|university_id|running_sum|
+----------+-----------+---+------+---------+---------------+-------------+-----------+
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|         20|
|      S005|Arjun Patel| 20|     M|Ahmedabad|           2022|          U02|         40|
|      S010|Ananya Iyer| 21|     F|  Chennai|           2021|          U01|         21|
|      S010|Ananya Iyer| 21|     F|  Chennai|           2021|          U01|         42|
|      S004|Priya Mehta| 22|     F|    Delhi|           2020|          U03|         22|
|      S001|Amit Sharma| 20|     M|    Delhi|           2022|          U01|         42|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|         21|
|      S006|Sneha Reddy| 21|     F|Hyderabad|           2021|          U03|         42|
|      S007|Vikas Gupta| 23|    

windowSpecs: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@b2a4482

In [68]:
spark.stop()